In [ ]:
# Interactive Figure
%matplotlib ipympl 

In [ ]:
import time
import numpy as np
from matplotlib import pyplot as plt
from qualang_tools.units import unit
u = unit(coerce_to_integer=True)
from qm.qua import *
from qm import SimulationConfig
from qualang_tools.loops import from_array
import QM

# Send pulses and observe them

In [ ]:
# This block is using QUA directives which are compiled to the FPGA
chunk_size = 5 # 20ns
readout_len = 800
IQ_size = readout_len//(chunk_size*4)
with program() as prog:
    # Variable declaration
    m = declare(int)
    I = declare(fixed, size=IQ_size)
    Q = declare(fixed, size=IQ_size)
    I_st = declare_stream()
    Q_st = declare_stream()
    # Start measurement
    measure("readout", "scope",
    demod.sliced("cos", I, chunk_size, "out1"),
    demod.sliced("minus_sin", Q, chunk_size, "out1"),
            )
    # Pulse sequence
    play('pulse','qubit',duration=100*u.ns)
    wait(200*u.ns)
    play('pulse','qubit',duration=100*u.ns)
                       
    with for_(m, 0, m < IQ_size, m + 1): 
        save(I[m], I_st)
        save(Q[m], Q_st)

    # Select data to be sent to the client 
    with stream_processing():
        I_st.buffer(IQ_size).save('I')
        Q_st.buffer(IQ_size).save('Q')

In [ ]:
# Run the code 
job = QM.Job(prog)

In [ ]:
# Plot
I,Q = job.get_results('I','Q')
t = np.arange(IQ_size)*chunk_size*4
fig,ax=plt.subplots()
ax.plot(t,I,t,Q)

# Exercises
- Link to the Quantum Machines documentation: https://docs.quantum-machines.co/latest/
- Read the documentation on [demodulation](https://docs.quantum-machines.co/latest/docs/Guides/demod/)

## Exercise 1: IQ Rotation
- Dephase the second pulse by $\pi/2$ using a [`frame_rotation`](https://docs.quantum-machines.co/latest/docs/API_references/qua/dsl_main/#for_docs.dsl.frame_rotation) command applied to the `qubit` element. Modify the delay between the pulses: what do you observe?
- Try different phase shifts in order to produce two pulses with negative $I$ values, one pulse with positive $I$ and the second with positive $Q$, etc.

## Exercise 2: Averaging
- Reduce the amplitude of the two pulses by a factor of 100 using the `amp` command inside `play`. Verify the result.

- Write a loop to repeat the entire sequence (measurement and pulse) 1000 times:
    ```
        n = declare(int)
        with for_(n, 0, n < 1000, n + 1):
            measure("readout", "scope",
            demod.sliced("cos", I, chunk_size, "out1"),
            demod.sliced("minus_sin", Q, chunk_size, "out1"),
                    )
            play('pulse','qubit',duration=100*u.ns)
            wait(200*u.ns)
            play('pulse','qubit',duration=100*u.ns)
                               
            with for_(m, 0, m < IQ_size, m + 1): 
                save(I[m], I_st)
                save(Q[m], Q_st) 
    ```
    You should obtain the same trace as before. In order to average over the loop repetitions, modify the stream processing block as follows:
  ```
      with stream_processing():
        I_st.buffer(IQ_size).average().save('I')
        Q_st.buffer(IQ_size).average().save('Q')
        
    ``` 
You can read the documentation on [stream buffering and averaging](https://docs.quantum-machines.co/latest/docs/Guides/stream_proc/#using-stream-operators).


**Remember to remove the phase rotation before averaging the traces**

- Replace one of the pulses with a Gaussian pulse (remove the `duration` keyword) and observe the result.
